In [10]:
import unicodedata
from bs4 import BeautifulSoup

def clean_description(html):
    # 1. No description -> return an empty string
    if html is None:
        return ""

    # 2. Parse the HTML
    soup = BeautifulSoup(html, "html.parser")

    # 3. Remove non-text tags completely: CSS and videos
    for tag in soup.find_all(["style", "iframe"]):
        tag.decompose()

    # 4. Extract the text, one piece of text per line
    text = soup.get_text("\n")

    # 5. Replace special characters (non-breaking space, zero-width space, BOM, carriage return)
    text = text.replace("\xa0", " ")
    text = text.replace(chr(8203), "")
    text = text.replace(chr(65279), "")
    text = text.replace("\r", "")

    # 6. Normalize Vietnamese characters to one Unicode form (NFC)
    text = unicodedata.normalize("NFC", text)

    # 7. Check each line: skip empty lines and the Tiki tax line
    result = []
    for line in text.split("\n"):
        line = line.strip()
        if line == "":
            continue
        if "Giá sản phẩm trên Tiki" in line:
            continue
        result.append(line)

    return "\n".join(result)

In [11]:
import json

P = json.load(open("data/products/products_001.json", encoding="utf-8"))
d = P[0]["description"]
print(d[:800])
print("=" * 60)
print(clean_description(d))

<ul>
<li><strong>Bộ Xếp Hình Gương Thu Tia Sáng (2035 Mảnh Ghép) </strong>được làm từ chất liệu giấy bồi cao cấp, bền đẹp không gây độc hại an toàn cho sức khỏe của bé.</li>
<li>Với màu sắc sinh động cùng ý tưởng về gương thu sẽ mang đến cho trẻ những giây phút trải nghiệm độc đáo, thú vị.</li>
<li>Kiểu dáng nhỏ gọn, dễ dàng mang theo. Bên cạnh đó, sau khi hoàn thành bộ xếp hình bé có thể đóng khung giấy và treo lên để trang trí phòng ngủ của mình.</li>
<li>Được thiết kế để hỗ trợ bé yêu của bạn phát triển trí não và các kỹ năng cần thiết khác ngay những năm tháng đầu đời.</li>
<li>Với bộ sản phẩm này, bé có thể phát huy khả năng tư duy và sáng tạo, rèn luyện tính kiên nhẫn, sự khéo léo, óc logic cũng như trí tưởng tượng của mình qua việc lắp ráp.</li>
</ul><p>Giá sản phẩm trên Tiki đã bao
Bộ Xếp Hình Gương Thu Tia Sáng (2035 Mảnh Ghép)
được làm từ chất liệu giấy bồi cao cấp, bền đẹp không gây độc hại an toàn cho sức khỏe của bé.
Với màu sắc sinh động cùng ý tưởng về gương thu sẽ mang 

In [12]:
count_tag = 0
count_tax = 0
count_empty = 0

for p in P:
    clean = clean_description(p["description"])
    if "<" in clean and ">" in clean:
        count_tag = count_tag + 1
    if "Giá sản phẩm trên Tiki" in clean:
        count_tax = count_tax + 1
    if clean == "":
        count_empty = count_empty + 1

print("Products:", len(P))
print("Still has tags:", count_tag)
print("Still has tax line:", count_tax)
print("Empty after cleaning:", count_empty)

Products: 689
Still has tags: 0
Still has tax line: 0
Empty after cleaning: 32


In [13]:
from pathlib import Path

Path("data/clean").mkdir(exist_ok=True)

for i in range(1, 201):
    name = f"products_{i:03d}.json"

    # read the raw file
    products = json.load(open("data/products/" + name, encoding="utf-8"))

    # clean the description of every product
    for p in products:
        p["description"] = clean_description(p["description"])

    # write to the new folder
    with open("data/clean/" + name, "w", encoding="utf-8") as f:
        json.dump(products, f, ensure_ascii=False)

    print("Done", name)

Done products_001.json
Done products_002.json
Done products_003.json
Done products_004.json
Done products_005.json
Done products_006.json
Done products_007.json
Done products_008.json
Done products_009.json
Done products_010.json
Done products_011.json
Done products_012.json
Done products_013.json
Done products_014.json
Done products_015.json
Done products_016.json
Done products_017.json
Done products_018.json
Done products_019.json
Done products_020.json
Done products_021.json
Done products_022.json
Done products_023.json
Done products_024.json
Done products_025.json
Done products_026.json
Done products_027.json
Done products_028.json
Done products_029.json
Done products_030.json
Done products_031.json
Done products_032.json
Done products_033.json
Done products_034.json
Done products_035.json
Done products_036.json
Done products_037.json
Done products_038.json
Done products_039.json
Done products_040.json
Done products_041.json
Done products_042.json
Done products_043.json
Done produc

In [15]:
raw_ids = []
clean_ids = []
count_tag = 0
count_tax = 0
count_empty = 0

for i in range(1, 201):
    name = f"products_{i:03d}.json"
    raw = json.load(open("data/products/" + name, encoding="utf-8"))
    clean = json.load(open("data/clean/" + name, encoding="utf-8"))

    for p in raw:
        raw_ids.append(p["id"])

    for p in clean:
        clean_ids.append(p["id"])
        d = p["description"]
        if "</" in d or "<p>" in d or "<br" in d or "<div" in d:
            count_tag = count_tag + 1
        if "Giá sản phẩm trên Tiki" in d:
            count_tax = count_tax + 1
        if d == "":
            count_empty = count_empty + 1

print("Raw products:", len(raw_ids))
print("Clean products:", len(clean_ids))
print("Same ids, same order:", raw_ids == clean_ids)
print("Still has tags:", count_tag)
print("Still has tax line:", count_tax)
print("Empty after cleaning:", count_empty)

Raw products: 124299
Clean products: 124299
Same ids, same order: True
Still has tags: 30
Still has tax line: 0
Empty after cleaning: 1744
